In [1]:
import sqlite3
import pandas as pd

conn = sqlite3.connect("../data/lending_club.db")

def q(sql):
    return pd.read_sql_query(sql, conn)

In [4]:
q("""
SELECT COUNT(*) AS loans,
       ROUND(AVG(is_default) * 100, 2) AS default_rate_pct
FROM loans
""")

,loans,default_rate_pct
0,733453,18.81


The LendingClub dataset contains 733,453 loans, with an overall default rate of 18.81%. This provides a baseline for evaluating how borrower and loan characteristics are associated with credit risk.

In [5]:
q("""
SELECT grade,
       COUNT(*) AS loans,
       ROUND(AVG(is_default) * 100, 2) AS default_rate_pct
FROM loans
GROUP BY grade
ORDER BY grade
""")

,grade,loans,default_rate_pct
0,A,125444,5.39
1,B,210969,11.96
2,C,205248,20.62
3,D,113354,28.75
4,E,55828,37.68
5,F,18324,43.49
6,G,4286,48.18


Default rates increase consistently as credit grade declines. Grade A loans have a 5.39% default rate, compared with 48.18% for Grade G loans. This represents a substantial difference in observed default rates across credit grades and suggests that credit grade is an important variable to consider when analyzing loan risk.

In [14]:
q("""
SELECT purpose, COUNT(*) AS loans,
       ROUND(AVG(is_default) * 100, 2) AS default_rate_pct
FROM loans
GROUP BY purpose
ORDER BY default_rate_pct DESC
""")

,purpose,loans,default_rate_pct
0,small_business,6616,27.49
1,renewable_energy,384,22.40
2,moving,4241,22.35
3,house,2730,21.68
4,debt_consolidation,437744,20.02
5,medical,6839,20.02
6,other,33771,19.96
7,major_purchase,12716,17.73
8,vacation,3917,17.44
9,home_improvement,42417,17.05


Default rates vary across loan purposes. Small business loans have the highest observed default rate at 27.49%, while car loans have a 14.60% default rate. Debt consolidation represents the largest segment of the portfolio with 437,744 loans and a 20.02% default rate.

The educational category contains only one loan, so its 0.00% default rate is not meaningful for comparison due to the extremely small sample size.

In [7]:
# By loan term
q("""
SELECT TRIM(term) AS term, COUNT(*) AS loans,
       ROUND(AVG(is_default) * 100, 2) AS default_rate_pct
FROM loans
GROUP BY TRIM(term)
""")

,term,loans,default_rate_pct
0,36 months,546018,14.07
1,60 months,187435,32.63


36-month loans have a 14.07% default rate, compared with 32.63% for 60-month loans. The 60-month loans therefore show a substantially higher observed default rate in this dataset.

Because loan term may be related to other borrower and loan characteristics, this relationship should be investigated alongside factors such as credit grade, interest rate, income, and debt-to-income ratio.

In [8]:
# By issue year
q("""
SELECT SUBSTR(issue_d, -4) AS issue_year, COUNT(*) AS loans,
       ROUND(AVG(is_default) * 100, 2) AS default_rate_pct
FROM loans
GROUP BY issue_year
ORDER BY issue_year
""")

,issue_year,loans,default_rate_pct
0,2013,134804,15.60
1,2014,223103,18.45
2,2015,375546,20.19


The observed default rate increased from 15.60% in 2013 to 20.19% in 2015, while loan volume also increased substantially from 134,804 to 375,546 loans. This indicates that the portfolio expanded while the observed default rate increased during this period.

Because default rates can be affected by changes in borrower mix, underwriting, loan characteristics, and the amount of time loans have had to mature, this trend should be interpreted as an observed pattern rather than evidence of causation.

In [9]:
q("""
SELECT CASE
         WHEN annual_inc < 40000 THEN '1. Under 40k'
         WHEN annual_inc < 60000 THEN '2. 40-60k'
         WHEN annual_inc < 80000 THEN '3. 60-80k'
         WHEN annual_inc < 120000 THEN '4. 80-120k'
         ELSE '5. 120k+'
       END AS income_band,
       COUNT(*) AS loans,
       ROUND(AVG(is_default) * 100, 2) AS default_rate_pct
FROM loans
GROUP BY income_band
ORDER BY income_band
""")

,income_band,loans,default_rate_pct
0,1. Under 40k,115045,22.18
1,2. 40-60k,199062,20.76
2,3. 60-80k,167917,19.04
3,4. 80-120k,163321,16.39
4,5. 120k+,88108,14.10


The observed default rate decreases as income increases, from 22.18% for borrowers earning under $40,000 to 14.10% for borrowers earning $120,000 or more. The results suggest that higher-income borrowers in this dataset experienced lower default rates.

Income alone does not determine credit risk, so this relationship should also be evaluated alongside factors such as credit grade, debt-to-income ratio, and loan term.

DATA QUALITY CHECK

In [10]:
df = q("SELECT * FROM loans")
df.isna().mean().sort_values(ascending=False).head(15)

emp_length          0.053956
revol_util          0.000473
dti                 0.000003
id                  0.000000
issue_date          0.000000
application_type    0.000000
total_acc           0.000000
revol_bal           0.000000
pub_rec             0.000000
open_acc            0.000000
fico_range_high     0.000000
fico_range_low      0.000000
earliest_cr_line    0.000000
delinq_2yrs         0.000000
addr_state          0.000000
dtype: float64

In [11]:
q("""
SELECT MIN(annual_inc) AS min_inc, MAX(annual_inc) AS max_inc,
       MIN(dti) AS min_dti, MAX(dti) AS max_dti,
       MAX(revol_util) AS max_revol_util
FROM loans
""")

,min_inc,max_inc,min_dti,max_dti,max_revol_util
0,0.0,9500000.0,0.0,999.0,892.3


In [18]:
q("SELECT COUNT(*) AS zero_income FROM loans WHERE annual_inc = 0")

,zero_income
0,2


In [15]:
q("SELECT COUNT(*) AS income_over_1m FROM loans WHERE annual_inc >= 1000000")

,income_over_1m
0,147


In [16]:
q("SELECT COUNT(*) AS dti_over_60 FROM loans WHERE dti > 60")

,dti_over_60
0,21


In [17]:
q("SELECT COUNT(*) AS revol_util_over_100 FROM loans WHERE revol_util > 100")

,revol_util_over_100
0,3064


CLEANING DATA

In [20]:
conn.execute("DROP TABLE IF EXISTS loans_clean")
conn.execute("""
CREATE TABLE loans_clean AS
SELECT *,
       CAST(SUBSTR(TRIM(term), 1, 2) AS INTEGER) AS term_months,
       SUBSTR(issue_d, -4) AS issue_year
FROM loans
WHERE annual_inc > 0
  AND annual_inc < 1000000
  AND dti >= 0
  AND dti <= 60
""")
conn.commit()
q("SELECT COUNT(*) AS rows_after_cleaning FROM loans_clean")

,rows_after_cleaning
0,733283
